Merges different datasets into one with the CSV structure text, generated.
Text is the text. Generated is an integer (0 = human text, 1 = AI generated text)

The AI_Human datasets is omitted (bad quality, require more cleaning)

In [142]:
# Load the individual datasets
import pandas as pd

ai_human = pd.read_csv("./data/datasets/AI_Human.csv")
ai_papers = pd.read_csv("./data/datasets/ai_papers.csv") # already correct format
ai_vs_human = pd.read_csv("./data/datasets/ai_vs_human_text.csv") # needs adjustment
ajur = pd.read_csv("./data/datasets/ajur.csv") # already correct format
balanced = pd.read_csv("./data/datasets/Balanced_AI_Human_Humanized_UPDATED.csv") # must be adjusted
detection = pd.read_csv("./data/datasets/ai_human_content_detection_dataset.csv") # must be adjusted
essays = pd.read_csv("./data/datasets/AI Generated Essays Dataset.csv") # already follows our structure
fivethousand = pd.read_csv("./data/datasets/5000human_5000machine.csv") # must be adjusted
intersect = pd.read_csv("./data/datasets/intersect.csv") # already correct format
medium = pd.read_csv("./data/datasets/ai_vs_human_dataset_medium.csv") # must be adjusted
prompts = pd.read_csv("./data/datasets/balanced_ai_human_prompts.csv") # already follows our structure
shuffled = pd.read_csv("./data/datasets/shuffled_data.csv") # must be adjusted
train = pd.read_csv("./data/datasets/train.csv") # must be adjusted
wikipedia = pd.read_csv("./data/datasets/wikipedia_gpt.csv") # must be adjusted

In [143]:
WITH_DATASET_EVALUATION = True

import nltk
nltk.download("punkt_tab")

def evaluate_ds(ds):
    if WITH_DATASET_EVALUATION:
        # Get human and AI texts
        h = ds.loc[ds["generated"] == 0, "text"].tolist()
        ai = ds.loc[ds["generated"] == 1, "text"].tolist()
        
        # Tokenize sentences
        s_h = [nltk.sent_tokenize(text) for text in h]
        s_ai = [nltk.sent_tokenize(text) for text in ai]
        
        # Flatten
        f_h = [item for sublist in s_h for item in sublist]
        f_ai = [item for sublist in s_ai for item in sublist]

        # Show number of sentences
        len_ai_t = len(f_ai)
        len_h_t = len(f_h)
        len_t = len_ai_t + len_h_t

        print(f"Number of human sentences: {len(f_h)} ({float(len_h_t) / float(len_t) * 100:.5f}%)")
        print(f"Number of AI sentences: {len(f_ai)} ({float(len_ai_t) / float(len_t) * 100:.5f}%)")

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [144]:
# Adjust AI/Human datasets
ai_human["generated"] = ai_human["generated"].astype(int)
ai_human.dropna(inplace=True)
human_texts = ai_human[ai_human["generated"] == 0]
human_texts = human_texts.sample(frac=0.02375, random_state=11)

evaluate_ds(human_texts)

Number of human sentences: 154846 (100.00000%)
Number of AI sentences: 0 (0.00000%)


In [145]:
# Adjust ai vs human texts
ai_vs_human = ai_vs_human[["text", "label"]]
ai_vs_human = ai_vs_human.rename(columns={"label":"generated"})
ai_vs_human["generated"] = ai_vs_human["generated"].replace({"ai": 1, "human": 0})
ai_vs_human.dropna(inplace=True)

evaluate_ds(ai_vs_human)

Number of human sentences: 727 (52.87273%)
Number of AI sentences: 648 (47.12727%)


/tmp/ipykernel_123033/2159182361.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ai_vs_human["generated"] = ai_vs_human["generated"].replace({"ai": 1, "human": 0})


In [146]:
# Adjust balanced dataset
balanced = balanced[["text", "generated"]]
balanced = balanced[balanced["generated"] != 2] # drop humanized rows
balanced.dropna(inplace=True)

evaluate_ds(balanced)

Number of human sentences: 6616 (56.63899%)
Number of AI sentences: 5065 (43.36101%)


In [147]:
# Adjust format of detection dataset
detection = detection[["text_content", "label"]]
detection = detection.rename(columns = {"text_content": "text", "label": "generated"})
detection.dropna(inplace=True)

evaluate_ds(detection)

Number of human sentences: 17128 (48.99874%)
Number of AI sentences: 17828 (51.00126%)


In [148]:
# Adjust format of fivethousand datasetevalaute_ds
fivethousand = fivethousand.rename(columns = {"label": "generated"})
fivethousand = fivethousand[["text", "generated"]]
fivethousand.dropna(inplace=True)

evaluate_ds(fivethousand)

Number of human sentences: 64170 (48.23144%)
Number of AI sentences: 68876 (51.76856%)


In [149]:
# Adjust format of medium dataset
medium = medium[["text", "label"]]
medium = medium.rename(columns = {"label": "generated"})
medium["generated"] = medium["generated"].replace({"ai": 1, "human": 0})
medium.dropna(inplace=True)

evaluate_ds(medium)

Number of human sentences: 360 (50.91938%)
Number of AI sentences: 347 (49.08062%)


/tmp/ipykernel_123033/2387865756.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  medium["generated"] = medium["generated"].replace({"ai": 1, "human": 0})


In [150]:
# Adjust shuffled dataset
shuffled.dropna(inplace=True)
shuffled = shuffled.rename(columns = {"Data": "text", "Labels": "generated"})

evaluate_ds(shuffled)

Number of human sentences: 67828 (63.09112%)
Number of AI sentences: 39680 (36.90888%)


In [151]:
# Adjust format of train dataset

## Cut out different subdatasets
# Human data
human = train[["Human_story"]]
human = human.rename(columns = {"Human_story": "text"})
human["generated"] = 0
human.dropna(inplace=True)

# Gemma data
gemma = train[["gemma-2-9b"]]
gemma = gemma.rename(columns = {"gemma-2-9b": "text"})
gemma["generated"] = 1
gemma.dropna(inplace=True)

# GPT data
gpt = train[["GPT_4-o"]]
gpt = gpt.rename(columns = {"GPT_4-o": "text"})
gpt["generated"] = 1
gpt.dropna(inplace=True)

# Llama data
llama = train[["llama-8B"]]
llama = llama.rename(columns = {"llama-8B": "text"})
llama["generated"] = 1
llama.dropna(inplace=True)

print("HUMAN")
evaluate_ds(human)

print("GEMMA")
evaluate_ds(gemma)

print("GPT")
evaluate_ds(gpt)

print("LLAMA")
evaluate_ds(llama)


HUMAN
Number of human sentences: 235549 (100.00000%)
Number of AI sentences: 0 (0.00000%)
GEMMA
Number of human sentences: 0 (0.00000%)
Number of AI sentences: 125588 (100.00000%)
GPT
Number of human sentences: 0 (0.00000%)
Number of AI sentences: 201886 (100.00000%)
LLAMA
Number of human sentences: 0 (0.00000%)
Number of AI sentences: 133566 (100.00000%)


In [152]:
# Adjust wikipedia dataset
# Get human texts
wikipedia_human = wikipedia[["human_text"]]
wikipedia_human = wikipedia_human.rename(columns={"human_text": "text"})
wikipedia_human["generated"] = 0
wikipedia_human.dropna(inplace=True)

wikipedia_gpt = wikipedia[["gpt_text"]]
wikipedia_gpt = wikipedia_gpt.rename(columns={"gpt_text": "text"})
wikipedia_gpt["generated"] = 1
wikipedia_gpt.dropna(inplace=True)

print("HUMAN")
evaluate_ds(wikipedia_human)

print("AI")
evaluate_ds(wikipedia_gpt)

HUMAN
Number of human sentences: 3312 (100.00000%)
Number of AI sentences: 0 (0.00000%)
AI
Number of human sentences: 0 (0.00000%)
Number of AI sentences: 5503 (100.00000%)


In [153]:
evaluate_ds(ai_papers)

Number of human sentences: 0 (0.00000%)
Number of AI sentences: 79023 (100.00000%)


In [154]:
evaluate_ds(ajur)

Number of human sentences: 48919 (100.00000%)
Number of AI sentences: 0 (0.00000%)


In [155]:
evaluate_ds(intersect)

Number of human sentences: 10318 (100.00000%)
Number of AI sentences: 0 (0.00000%)


In [156]:
evaluate_ds(essays)

Number of human sentences: 38032 (97.94237%)
Number of AI sentences: 799 (2.05763%)


In [157]:
evaluate_ds(prompts)

Number of human sentences: 38032 (94.79325%)
Number of AI sentences: 2089 (5.20675%)


In [158]:
# Merge individual datasets into one large dataset
merged = pd.concat([human_texts, 
    ai_papers,
    ai_vs_human,
    ajur,
    balanced,
    essays,
    detection,
    fivethousand,
    gemma,
    gpt,
    human,
    intersect,
    llama,
    medium,
    prompts,
    shuffled,
    wikipedia_gpt,
    wikipedia_human], ignore_index=True)

In [159]:
# Split data of new dataset into human and AI texts
human_texts = merged.loc[merged["generated"] == 0, "text"].tolist()
ai_texts = merged.loc[merged["generated"] == 1, "text"].tolist()

In [160]:
# Tokenize texts (split into list of individual sentences)
import nltk
nltk.download("punkt_tab")

sentence_tokenized_human_texts = [nltk.sent_tokenize(text) for text in human_texts]
sentence_tokenized_ai_texts = [nltk.sent_tokenize(text) for text in ai_texts]

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [161]:
# Flatten lists (split list of list of sentences into list of sentences)
flat_human_texts = [item for sublist in sentence_tokenized_human_texts for item in sublist]
flat_ai_texts = [item for sublist in sentence_tokenized_ai_texts for item in sublist]

In [162]:
# Show number of sentences
len_ai_t = len(flat_ai_texts)
len_h_t = len(flat_human_texts)
len_t = len_ai_t + len_h_t

print(f"Number of human sentences: {len(flat_human_texts)} ({float(len_h_t) / float(len_t) * 100:.2f}%)")
print(f"Number of AI sentences: {len(flat_ai_texts)} ({float(len_ai_t) / float(len_t) * 100:.2f}%)")

Number of human sentences: 685837 (50.18%)
Number of AI sentences: 680898 (49.82%)


In [163]:
# Combine
import numpy as np

all_texts = flat_human_texts + flat_ai_texts
labels = np.concatenate([
    np.zeros(len_h_t, dtype=int),
    np.ones(len_ai_t, dtype=int)
])

In [164]:
# Split into training, testing and validation data
# Train: 70%, Test: 15%, Validation: 15%
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(
    all_texts, labels, test_size=0.3, random_state=7, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=7, stratify=y_tmp
)

In [165]:
# Store train dataset
import pandas as pd
from local_utilities.dataset import get_path_train

df = pd.DataFrame({
    "text": X_train,
    "generated": y_train
})
df.to_csv(get_path_train(), index=False)

In [166]:
# Store test dataset
from local_utilities.dataset import get_path_test

df = pd.DataFrame({
    "text":  X_test,
    "generated": y_test
})
df.to_csv(get_path_test(), index=False)

In [167]:
# Store validation dataset
from local_utilities.dataset import get_path_validation

df = pd.DataFrame({
    "text":  X_val,
    "generated": y_val
})
df.to_csv(get_path_validation(), index=False)

In [168]:
# Get sizes

print(f"Train: Human: {(y_train == 0).sum()}, AI: {(y_train == 1).sum()}, Human-%: {float((y_train == 0).sum()) / len(y_train)*100:.2f}, total: {len(y_train)}")
print(f"Test: Human: {(y_test == 0).sum()}, AI: {(y_test == 1).sum()}, Human-%: {float((y_test == 0).sum()) / len(y_test)*100:.2f}, total: {len(y_test)}")
print(f"Validate: Human: {(y_val == 0).sum()}, AI: {(y_val == 1).sum()}, Human-%: {float((y_val == 0).sum()) / len(y_val)*100:.2f}, total: {len(y_val)}")

Train: Human: 480086, AI: 476628, Human-%: 50.18, total: 956714
Test: Human: 102876, AI: 102135, Human-%: 50.18, total: 205011
Validate: Human: 102875, AI: 102135, Human-%: 50.18, total: 205010
